In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [3]:
spark = SparkSession.builder.appName("case-study-notes").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 03:47:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
customers_df = spark.read.csv("Data/customers.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("Data/order_items.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("Data/orders.csv", header=True, inferSchema=True)
products_df = spark.read.csv("Data/products.csv", header=True, inferSchema=True)
returns_df = spark.read.csv("Data/returns.csv", header=True, inferSchema=True)

26/06/16 03:47:12 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [5]:
customers_df.createOrReplaceTempView("customers")
order_items_df.createOrReplaceTempView("order_items")
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")
returns_df.createOrReplaceTempView("returns")

In [6]:
customers_df.show(5)
order_items_df.show(5)
orders_df.show(5)
products_df.show(5)
returns_df.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|selling_price|
+-------------+--------+----------+--------+-------------+
|            1|  227444|     28849|       5|       727.98|
|            2|   32708|     25471|  

In [7]:
sql_df1 = spark.sql("""
    SELECT 
        (SELECT count(customer_id) FROM customers) as total_customers,
        (SELECT count(product_id) FROM products) as total_products,
        (SELECT count(order_id) FROM orders) as total_orders,
        (SELECT count(order_item_id) FROM order_items) as total_order_items,
        (SELECT count(return_id) FROM returns) as total_returns
""")

sql_df1.show()

+---------------+--------------+------------+-----------------+-------------+
|total_customers|total_products|total_orders|total_order_items|total_returns|
+---------------+--------------+------------+-----------------+-------------+
|         100000|         50000|     1000000|          3000000|       100000|
+---------------+--------------+------------+-----------------+-------------+



In [8]:
sql_df1.write.mode("overwrite").csv("output/result1", header = True)

In [9]:
sql_df2 = spark.sql("""
    SELECT category as product_category, 
    sum(unit_cost) as total_sales_amount
    from products
    group by category
""")

sql_df2.show()

+----------------+------------------+
|product_category|total_sales_amount|
+----------------+------------------+
|  Home & Kitchen| 2901364.330000004|
|          Sports| 2853163.040000003|
|     Electronics|2864604.7399999946|
|        Clothing| 2841424.610000002|
|           Books|2853871.8500000075|
|          Beauty|2919388.7500000037|
|            Toys|2851913.1100000013|
+----------------+------------------+



In [10]:
sql_df2.write.mode("overwrite").csv("output/result2", header = True)

In [11]:
sql_df3 = spark.sql("""
    SELECT  c.customer_id, 
    c.customer_name as customer,
    round(sum(oi.quantity*oi.selling_price),2) as total_purchase_amount
    FROM
    customers c 
    join orders o
    on c.customer_id = o.customer_id
    join order_items oi
    on o.order_id = oi.order_id
    where o.order_status = 'Delivered'
    group by c.customer_id,customer
    order by total_purchase_amount DESC
    limit 10

""")

sql_df3.show()

[Stage 58:>                                                         (0 + 2) / 2]

+-----------+--------------+---------------------+
|customer_id|      customer|total_purchase_amount|
+-----------+--------------+---------------------+
|      64560|Customer_64560|            119030.04|
|      65135|Customer_65135|            111137.38|
|      52275|Customer_52275|            108098.33|
|      28584|Customer_28584|            107848.24|
|      37277|Customer_37277|            107444.08|
|      17810|Customer_17810|            105355.56|
|      11201|Customer_11201|             104247.4|
|      33876|Customer_33876|             102933.8|
|      10188|Customer_10188|            100520.46|
|      97963|Customer_97963|             99773.07|
+-----------+--------------+---------------------+



In [12]:
sql_df3.write.mode("overwrite").csv("output/result3", header = True)

In [13]:
sql_df4 = spark.sql("""
    WITH LatestYear AS (
        SELECT MAX(YEAR(order_date)) as max_year 
        FROM orders
    )
    
    SELECT 
        MONTH(o.order_date) as sales_month,
        ROUND(SUM(oi.quantity * oi.selling_price), 2) as total_revenue
    FROM orders o
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    JOIN LatestYear ly 
        ON YEAR(o.order_date) = ly.max_year
    WHERE 
        o.order_status = 'Delivered'
    GROUP BY 
        MONTH(o.order_date)
    ORDER BY 
        sales_month ASC
""")

sql_df4.show()

[Stage 81:>                                                         (0 + 2) / 2]

+-----------+--------------+
|sales_month| total_revenue|
+-----------+--------------+
|          1|2.2365795123E8|
|          2| 2.077295194E8|
|          3|2.2339561183E8|
|          4|2.1220716088E8|
|          5|2.2272812783E8|
|          6|2.1422183647E8|
|          7|2.2192965891E8|
|          8|2.2081820508E8|
|          9|2.1698195885E8|
|         10|2.1992173365E8|
|         11|2.1664222997E8|
|         12|2.2202115806E8|
+-----------+--------------+



In [14]:
sql_df4.write.mode("overwrite").csv("output/result4", header = True)

In [24]:
sql_df5 = spark.sql("""
   select p.category as product_category, 
   count(distinct o.order_id) as total_orders,
   count(distinct r.order_id) as returned_orders,
   round((returned_orders*100/total_orders),2) as return_percentage
   from products p 
   join order_items oi
   on p.product_id = oi.product_id
   join orders o
   on oi.order_id = o.order_id
   left join returns r         
   on r.order_id = o.order_id
   group by p.category
   order by return_percentage
""")

sql_df5.show()

[Stage 181:>                                                        (0 + 2) / 2]

+----------------+------------+---------------+-----------------+
|product_category|total_orders|returned_orders|return_percentage|
+----------------+------------+---------------+-----------------+
|        Clothing|      347805|          34680|             9.97|
|     Electronics|      346773|          34744|            10.02|
|           Books|      347520|          34832|            10.02|
|          Beauty|      349992|          35070|            10.02|
|  Home & Kitchen|      351594|          35260|            10.03|
|          Sports|      345930|          34699|            10.03|
|            Toys|      349584|          35112|            10.04|
+----------------+------------+---------------+-----------------+



In [25]:
sql_df5.write.mode("overwrite").csv("output/result5", header = True)

In [28]:
sql_df6 = spark.sql("""
WITH payment_counts AS (
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS used_times
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.state, o.payment_mode
),

ranked_modes AS (
    SELECT *,
           RANK() OVER (
               PARTITION BY state
               ORDER BY used_times DESC
           ) AS rnk
    FROM payment_counts
)

SELECT
    state,
    payment_mode,
    used_times
FROM ranked_modes
WHERE rnk = 1
""")

sql_df6.show()

[Stage 202:>                                                        (0 + 2) / 2]

+-----+----------------+----------+
|state|    payment_mode|used_times|
+-----+----------------+----------+
|   CA|             UPI|     20246|
|   FL|      Debit Card|     20010|
|   GA|     Net Banking|     20041|
|   IL|Cash on Delivery|     20498|
|   MI|      Debit Card|     20416|
|   NC|     Net Banking|     19596|
|   NY|      Debit Card|     20369|
|   OH|     Net Banking|     20351|
|   TX|             UPI|     20065|
|   WA|             UPI|     20244|
+-----+----------------+----------+



In [37]:
sql_df6.write.mode("overwrite").csv("output/result6", header = True)

In [36]:
sql_df7 = spark.sql("""
    SELECT  c.customer_id,
            c.customer_name as customer,
            count(distinct p.category) as categories_selected,
            round(sum(oi.quantity*oi.selling_price),2) as total_spent
        FROM
        customers c 
        join orders o
             on c.customer_id = o.customer_id
        join order_items oi
             on o.order_id = oi.order_id
        join products p
             on p.product_id = oi.product_id
        group by
             c.customer_id,customer
        having
            total_spent > 100000
            and categories_selected >= 5
        
""")
sql_df7.show()

26/06/16 05:52:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 05:52:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 234:>                                                        (0 + 2) / 2]

+-----------+--------------+-------------------+-----------+
|customer_id|      customer|categories_selected|total_spent|
+-----------+--------------+-------------------+-----------+
|      26241|Customer_26241|                  7|   121047.8|
|      52297|Customer_52297|                  7|  107812.68|
|      41157|Customer_41157|                  7|   105187.1|
|      84144|Customer_84144|                  7|  103326.02|
|      46860|Customer_46860|                  7|  100298.43|
|      81900|Customer_81900|                  7|  127465.68|
|      17271|Customer_17271|                  7|  103096.35|
|      97920|Customer_97920|                  7|  117591.17|
|      90203|Customer_90203|                  7|  109729.63|
|      18149|Customer_18149|                  7|   101780.7|
|      46060|Customer_46060|                  7|  115805.04|
|      90970|Customer_90970|                  7|  111172.63|
|      17979|Customer_17979|                  7|  132436.51|
|      60321|Customer_60

In [38]:
sql_df7.write.mode("overwrite").csv("output/result7", header = True)

26/06/16 05:53:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 05:53:19 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [47]:
sql_df8 = spark.sql("""
    with product_revenue as(
    SELECT 
    p.category as category,
    p.product_name,
    ROUND(SUM(oi.quantity * oi.selling_price), 2) as total_revenue
    from products p
    join order_items oi
      on p.product_id = oi.product_id
    group by 
    p.category,
    p.product_name
    ),
    
    product_ranks as(
    select category,
    product_name,
    total_revenue,
    rank() over(
    partition by category
    order by total_revenue desc) as rnk
    from 
    product_revenue
    )

    select category,
    product_name,
    total_revenue
    from 
    product_ranks
    where rnk <= 3
    order by category,
    total_revenue desc
    
    
""")
sql_df8.show()

[Stage 274:>                                                        (0 + 2) / 2]

+--------------+-------------+-------------+
|      category| product_name|total_revenue|
+--------------+-------------+-------------+
|        Beauty|Product_44016|    277567.99|
|        Beauty|Product_14849|     274894.2|
|        Beauty|  Product_786|     272174.7|
|         Books|Product_35314|    296468.78|
|         Books|Product_28311|    286757.72|
|         Books|Product_37479|    276736.71|
|      Clothing| Product_7025|    293821.97|
|      Clothing| Product_1560|    288474.09|
|      Clothing|Product_31322|    282241.17|
|   Electronics| Product_6719|    299113.87|
|   Electronics|Product_23519|    289561.72|
|   Electronics|Product_38170|    288875.23|
|Home & Kitchen| Product_5012|    305836.22|
|Home & Kitchen|Product_37452|    286817.42|
|Home & Kitchen|Product_27682|    283340.36|
|        Sports|Product_41700|    297548.24|
|        Sports|Product_33334|    279540.94|
|        Sports|Product_32124|    277392.05|
|          Toys| Product_4967|     298987.2|
|         

In [ ]:
sql_df7.write.mode("overwrite").csv("output/result7", header = True)